# Remove background

## Sources:
- [Background Subtraction – OpenCV 3.4 with python 3 Tutorial 32](https://pysource.com/2018/05/17/background-subtraction-opencv-3-4-with-python-3-tutorial-32/)
- [Background Removal Tutorial using OpenCV - Python | Replace Background with Video - WITH CODE](https://www.youtube.com/watch?v=tKA_Ht4P1Gw&t=398s)

## Import modules

In [1]:
# import internal modules
# from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
from pathlib import Path
from datetime import date


# import 3rd-party modules
import cv2
import numpy as np

# # import local modules
# from utils.renderer.resizer import resize_with_pad, resize_with_crop
# from utils.project_manager import Project

In [7]:
# get current date
today = date.today().strftime("%Y%m%d")

# set input & output video path
video_path = Path("/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/images/db/P1899019.mov")
out_path = f"assets/images/bg_remover/{video_path.stem}_{today}.mp4"

# Initialize video stream
video_cap = cv2.VideoCapture(str(video_path))

# set window names
WINDOW_NAME_DIFF = "diff"
WINDOW_NAME_VIDEO = "video"

# get video parameters
video_n_frames = video_cap.get(cv2.CAP_PROP_FRAME_COUNT)
video_fps = video_cap.get(cv2.CAP_PROP_FPS)
video_width = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"number of frames = {video_n_frames}")
print(f"fps = {video_fps}")
print(f"video width = {video_width}")
print(f"video height = {video_height}")

# set frame position to a given index
video_cap.set(cv2.CAP_PROP_POS_FRAMES, int(17*video_fps)) # at 17 seconds

# read frame at this index
_, ref_frame = video_cap.read()
# ref_img_height, ref_img_width, ref_img_channel = ref_frame.shape
ref_gray = cv2.cvtColor(ref_frame, cv2.COLOR_BGR2GRAY)
ref_gray = cv2.GaussianBlur(ref_gray, (5, 5), 0)

# set frame position back to the start
video_cap.set(cv2.CAP_PROP_POS_FRAMES, 0) # at 17 seconds

subtractor = cv2.createBackgroundSubtractorMOG2(history=20, varThreshold=60, detectShadows=True)

# while accessing stream 
while video_cap.isOpened():

    # read video stream
    ret, frame = video_cap.read()

    # break out of loop if no more frame
    if not ret:
        print("Ignoring empty camera frame.")
        break

    mask = subtractor.apply(frame)

    # gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # gray_frame = cv2.GaussianBlur(gray_frame, (5, 5), 0)

    # difference = cv2.absdiff(ref_gray, gray_frame)
    # _, difference = cv2.threshold(difference, 25, 255, cv2.THRESH_BINARY)

    #  # create a mask
    # diff1 = cv2.subtract(frame,ref_frame)
    # diff2=cv2.subtract(ref_img,img)
    # diff = diff1+diff2
    # diff[abs(diff)<13.0]=0
    # gray = cv2.cvtColor(diff.astype(np.uint8), cv2.COLOR_BGR2GRAY)
    # gray[np.abs(gray) < 10] = 0
    # fgmask = gray.astype(np.uint8)
    # fgmask[fgmask>0]=255
    # #invert the mask
    # fgmask_inv = cv2.bitwise_not(fgmask)
    # #use the masks to extract the relevant parts from FG and BG
    # fgimg = cv2.bitwise_and(img,img,mask = fgmask)
    # bgimg = cv2.bitwise_and(bg,bg,mask = fgmask_inv)
    # #combine both the BG and the FG images
    # dst = cv2.add(bgimg,fgimg)
    # cv2.imshow('Background Removal',dst)

    # show to screen
    cv2.imshow(WINDOW_NAME_DIFF, mask)
    cv2.imshow(WINDOW_NAME_VIDEO, frame)

    # wait for a key 
    # 0xFF to check what key we pressed on the keyboard
    key = cv2.waitKey(10) & 0xFF

    # break out of the stream loop if esc is pressed
    if key == 27 or key == ord('q'):        
        break


# quit windows
cv2.destroyAllWindows()
cv2.waitKey(1) # workaround to effectively close window on mac

# release video stream
video_cap.release()

number of frames = 852.0
fps = 25.0
video width = 1920
video height = 1080
Ignoring empty camera frame.


In [ ]:
# get current date
today = date.today().strftime("%Y%m%d")

# set video path
filepath = Path("/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/images/db/P1899019.mov")
out_path = f"assets/images/bg_remover/{filepath.stem}_{today}.mp4"

codec = "H264"

# initialize video stream
cap = cv2.VideoCapture(str(filepath))

# set frame position to the index
cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)

_, first_frame = cap.read()
ref_img_height, ref_img_width, ref_img_channel = first_frame.shape

# create a videoWriter object
fourcc = cv2.VideoWriter_fourcc(*codec)
out_video = cv2.VideoWriter(filename=out_path, fourcc=fourcc, fps=fps, frameSize=(ref_img_width, ref_img_height))

# create empty list of frames
frames = []
frame_nb = 0

while True:

    # read frame
    ret, frame = cap.read()

    if not ret:
        break

    # append frame to list of frames
    frames.append(frame)

    if frame_nb%frame_factor == 0:
        # compute average of frames
        # bg_frame = np.mean(frames, axis=0).astype(dtype=np.uint8)
        out_frame = np.median(frames, axis=0).astype(dtype=np.uint8)

        # write output frame
        out_video.write(out_frame)

        frames = []

    frame_nb += 1

# release video stream & video rendering
cap.release()
out_video.release()